# Nuisance-ladder study: does the result depend on how many modeling knobs are floated?

The production fits float the five interaction-model knobs whose $+1\sigma$ injection biases
$F_A(Q^2=0.5~\mathrm{GeV}^2)$ by more than $0.2\,\sigma[F_A]$ (notebook 19, Section 8); every other
knob is a covariance term. A reviewer pointed out that the value of that cut is a choice.
`xml/nuisance_ladder_study/` tests it by floating the next knobs of the injection ranking,
cumulatively (`python/scripts/make_nuisance_ladder_study_xmls.py`):

| variant | knob added to the previous step | injection bias, $M_A$ fit |
|---|---|---|
| `n5` | none: the production set (RPA_CCQE, XSecShape_CCMEC, MFP_N, DecayAngMEC, FrCEx_N) | |
| `n6` | VecFFCCQEshape (two-point switch, restricted to $[0,1]$) | $-0.14$ |
| `n7` | FracPN_CCMEC | $+0.12$ |
| `n8` | xsr_scc_Fv3_SCC | $-0.10$ |
| `n9` | FrAbs_N | $-0.10$ |
| `n10` | NormCCMEC | $-0.04$ |

for the fits `ma` and `minerva_k6_nuisance` on three data sets, each answering one question:

* **asimov**: what floating costs in $\sigma[F_A]$, and whether the data constrain the added knob
  at all (posterior width over prior width);
* **opendata**: whether $F_A$ moves and whether the added knobs are pulled;
* **nuwro**: whether the added knobs absorb part of the known model mismatch.

For a knob with a linear response, floating it under a unit Gaussian prior is equivalent to keeping
it in the covariance: same $\chi^2_{\min}$, same best-fit $F_A$, same uncertainty. Only the
bookkeeping changes (an explicit pull, $\chi^2$ moved from the syst term to the pull term). The
ladder therefore measures the departures from that: non-linear spline response, the $[0,1]$ bound
of two-point switches, and the finite-throw covariance.

**Decision rule, fixed on 2026-11-16 before any fit was run.** Keep the production set if, in
every family and fit, (i) the best-fit $F_A(Q^2_{\rm ref})$ of every step through `n10` stays within
$0.2\,\sigma[F_A]$ of `n5`, and (ii) no added knob is pulled beyond $1\sigma$. Otherwise adopt the
smallest set after which further additions satisfy the rule.

Shifts are taken from the profile best fit (free of MCMC noise); widths and pulls from the chains.
Fits that have not finished are listed and skipped, so the notebook can be rerun while the study
is running.

In [ ]:
from dataclasses import replace
from pathlib import Path
import os
import re
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

# Locate ma_zexp whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
REPO = next(
    (parent / 'ma_zexp' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts/postfit_physical_parameters.py')
if str(REPO / 'python' / 'scripts') not in sys.path:
    sys.path.insert(0, str(REPO / 'python' / 'scripts'))

from postfit_physical_parameters import (NUISANCE_DISPLAY_NAMES, NUISANCE_SHORT_LABELS, NUWRO_INPUT_MA,
                                         PUBLICATION_RC, SPECS, _fa_curves, load_fit)
import make_nuisance_ladder_study_xmls as ladder

mpl.rcParams.update(PUBLICATION_RC)
SPEC = {spec.key: spec for spec in SPECS}

## Controls

`NUISANCE_LADDER_STUDY_ROOT` overrides the location of the fit outputs. `SHIFT_LIMIT` and
`PULL_LIMIT` are the two numbers of the decision rule.

In [ ]:
OUTPUT_ROOT = Path(os.environ.get('AXIAL_MASS_OUTPUT_ROOT', '/nevis/hopper/data/epelaez/axial_mass_cnpmu_iterfrozen'))
STUDY_ROOT = Path(os.environ.get('NUISANCE_LADDER_STUDY_ROOT', OUTPUT_ROOT / 'nuisance_ladder_study_fit_results'))
FAMILIES = ('asimov', 'opendata', 'nuwro')
FITS = list(ladder.FITS)                       # ma, minerva_k6_nuisance
VARIANTS = list(ladder.VARIANTS)               # n5 .. n10
REF = VARIANTS[0]                              # the production set
Q2_REFERENCE = 0.50                            # GeV^2, the comparison point of the injection study
Q2 = np.linspace(0.0, 2.0, 201)
IREF = int(np.argmin(np.abs(Q2 - Q2_REFERENCE)))
IN_RANGE = (Q2 > 0.05) & (Q2 <= 1.0)
SHIFT_LIMIT = 0.2                              # sigma[F_A] of n5
PULL_LIMIT = 1.0                               # prior sigma of the added knob
MAX_SAMPLES = 100_000                          # chain samples used for the F_A bands
BURN_IN, THIN = 0, 1
FIG_DIR = REPO / 'figs' / 'nuisance_ladder_study'
TABLE_DIR = REPO / 'tables' / 'nuisance_ladder_study'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

label_of = lambda branch: branch.removesuffix('_UBGenie')                  # PROfit plotname of a knob
display_of = lambda label: NUISANCE_DISPLAY_NAMES[label]                    # '<knob> pull', the key of the result arrays
short_of = lambda label: NUISANCE_SHORT_LABELS.get(display_of(label), label)
BASELINE = [label_of(b) for b in ladder.BASELINE]
ADDED = {v: [label_of(b) for b, _ in ladder.VARIANTS[v]] for v in VARIANTS}  # cumulative
LADDER_KNOBS = ADDED[VARIANTS[-1]]
TWO_POINT = {label_of(b) for b, two_point in ladder.LADDER if two_point}
N_FLOATED = {v: len(BASELINE) + len(ADDED[v]) for v in VARIANTS}
STEP_LABEL = {v: ('production' if v == REF else '+ ' + short_of(ADDED[v][-1])) for v in VARIANTS}

FAMILY_LABEL = {'asimov': 'Asimov', 'opendata': 'Open data', 'nuwro': 'NuWro fake data'}
FIT_LABEL = {'ma': r'$M_A$, GENIE prior', 'minerva_k6_nuisance': r'$\bar{\nu}$–H with nuisance, $k_{\max}=6$'}
FIT_COLOR = {'ma': '#D55E00', 'minerva_k6_nuisance': '#0072B2'}
FIT_MARKER = {'ma': 'o', 'minerva_k6_nuisance': 's'}
VARIANT_COLOR = {v: ('black' if v == REF else mpl.colormaps['viridis']((i - 1) / max(len(VARIANTS) - 2, 1)))
                 for i, v in enumerate(VARIANTS)}
print(f'study at {STUDY_ROOT}')
for v in VARIANTS:
    print(f'  {v:4s} {N_FLOATED[v]:2d} floated knobs  {STEP_LABEL[v]}')

## 1. Inventory

A fit counts as finished when its `*_v1_PROfile.root` exists. Every comparison is against `n5` of
the same family and fit, so a family without a finished `n5` shows no shifts yet.

In [ ]:
def fit_dir(family, variant, fit):
    return STUDY_ROOT / family / variant / fit


def profile_root(family, variant, fit):
    matches = sorted(fit_dir(family, variant, fit).glob('*_v1_PROfile.root'))
    return matches[0] if len(matches) == 1 else None


FINISHED = {(family, v, fit): profile_root(family, v, fit) is not None
            for family in FAMILIES for v in VARIANTS for fit in FITS}
INVENTORY = pd.DataFrame({v: {(family, fit): ('done' if FINISHED[family, v, fit] else '')
                              for family in FAMILIES for fit in FITS} for v in VARIANTS})
INVENTORY.index.names = ['family', 'fit']
display(INVENTORY)
print(f'{sum(FINISHED.values())} of {len(FINISHED)} fits finished')
if not any(FINISHED.values()):
    raise RuntimeError(f'no finished fit below {STUDY_ROOT}; run xml/nuisance_ladder_study/run_all.sh')

## 2. Posterior chains

`load_fit` is given the production `FitSpec` extended with the knobs floated in the variant, so
the added knobs are read from the chain and from `global_fit_result` like the production five.
The effective sample size is printed; shifts of chain medians are only meaningful above the MCMC
noise $\sqrt{1/\mathrm{ESS}_{\rm ref}+1/\mathrm{ESS}}$ (in units of $\sigma$), listed in Section 3.

In [ ]:
def ladder_spec(fit, variant):
    branches = tuple(branch for branch, _ in ladder.VARIANTS[variant])
    labels = tuple(label_of(branch) for branch in branches)
    spec = SPEC[fit]
    if spec.prior is None:      # dipole fit: the knobs follow M_A in profile_labels / chain_branches
        return replace(spec, profile_labels=spec.profile_labels + labels, chain_branches=spec.chain_branches + branches)
    return replace(spec, nuisance_labels=spec.nuisance_labels + labels, nuisance_branches=spec.nuisance_branches + branches)


RESULTS = {}
for family in FAMILIES:
    for fit in FITS:
        for v in VARIANTS:
            if not FINISHED[family, v, fit]:
                continue
            try:
                result = load_fit(ladder_spec(fit, v), suite=v, burn_in=BURN_IN, thin=THIN, data_root=STUDY_ROOT / family)
            except Exception as error:      # a file still being written, or a chain without the expected branches
                print(f'{family:9s} {v:4s} {fit:20s} skipped: {type(error).__name__}: {error}')
                continue
            RESULTS[family, v, fit] = result
            flag = '' if result['ess_min'] >= 2000 else '   <-- ESS below 2000'
            print(f'{family:9s} {v:4s} {fit:20s} {len(result["samples"]):>8,} samples, min ESS {result["ess_min"]:>8,.0f}{flag}')

## 3. $F_A$: shift and precision along the ladder

Per family, fit and variant, relative to `n5`:

* **best-fit shift**: $(F_A^{\rm bf}-F_A^{\rm bf,\,n5})/\sigma[F_A]_{n5}$ at $Q^2_{\rm ref}$, from the
  profile best fit; this is the number of the decision rule. `max |shift|` is its maximum over
  $0.05<Q^2\le1~\mathrm{GeV}^2$;
* **median shift**: the same from the chain medians, with its MCMC noise;
* **$\sigma$ ratio**: $\sigma[F_A]/\sigma[F_A]_{n5}$ from the chains, the precision cost of floating;
* for NuWro, **bias vs input**: $(F_A^{\rm bf}-F_A^{\rm NuWro})/\sigma[F_A]$ against the dipole the
  fake data were generated with ($M_A=1.03$ GeV).

The grey band in the figure is the $\pm0.2\,\sigma$ of the decision rule.

In [ ]:
def fa_band(result):
    curves = _fa_curves(result, Q2, max_samples=MAX_SAMPLES)
    q16, median, q84 = np.percentile(curves, [16, 50, 84], axis=0)
    best = _fa_curves({'spec': result['spec'], 'samples': np.atleast_2d(result['profile'])}, Q2)[0]
    return dict(best=best, median=median, q16=q16, q84=q84, mean=curves.mean(axis=0), sigma=curves.std(axis=0))


NUWRO_TRUTH = -1.2723 / (1 + Q2 / NUWRO_INPUT_MA ** 2) ** 2      # F_A < 0, the convention of _fa_curves
BANDS = {key: fa_band(result) for key, result in RESULTS.items()}

rows = []
for (family, v, fit), band in BANDS.items():
    ref, ref_result = BANDS.get((family, REF, fit)), RESULTS.get((family, REF, fit))
    row = dict(family=family, fit=fit, variant=v, n_floated=N_FLOATED[v], step=STEP_LABEL[v],
               ess_min=RESULTS[family, v, fit]['ess_min'],
               **{'delta_F [%]': 100 * band['sigma'][IREF] / abs(band['mean'][IREF])})
    if ref is not None:
        shift = (band['best'] - ref['best']) / ref['sigma']
        row.update({'best-fit shift': shift[IREF], 'max |shift|': np.abs(shift[IN_RANGE]).max(),
                    'median shift': (band['median'][IREF] - ref['median'][IREF]) / ref['sigma'][IREF],
                    'MCMC noise': np.sqrt(1 / ref_result['ess_min'] + 1 / row['ess_min']) if v != REF else 0.0,
                    'sigma ratio': band['sigma'][IREF] / ref['sigma'][IREF]})
    if family == 'nuwro':
        row['bias vs input'] = (band['best'][IREF] - NUWRO_TRUTH[IREF]) / band['sigma'][IREF]
    rows.append(row)
ORDER = pd.MultiIndex.from_product([FAMILIES, FITS, VARIANTS], names=['family', 'fit', 'variant'])
FA = pd.DataFrame(rows).set_index(['family', 'fit', 'variant']).sort_index()
display(FA.drop(columns='n_floated').reindex(ORDER).dropna(how='all').style.format({c: '{:.3f}' for c in FA.columns if FA[c].dtype.kind == 'f'} | {'ess_min': '{:,.0f}'}, na_rep=''))

families = [f for f in FAMILIES if f in FA.index.get_level_values('family')]
panels = [('best-fit shift', rf'best-fit $F_A$ shift at $Q^2={Q2_REFERENCE:g}$ [$\sigma_{{n5}}$]'),
          ('sigma ratio', r'$\sigma[F_A]\,/\,\sigma[F_A]_{n5}$')]
x = np.arange(len(VARIANTS))
fig, axes = plt.subplots(len(panels), len(families), figsize=(4.3 * len(families), 6.2), sharex=True, sharey='row', squeeze=False)
for col, family in enumerate(families):
    for row, (column, ylabel) in enumerate(panels):
        ax = axes[row][col]
        for k, fit in enumerate(FITS):
            if (family, fit) not in FA.droplevel('variant').index or column not in FA:
                continue
            part = FA.loc[(family, fit)].reindex(VARIANTS)
            ax.plot(x + (k - 0.5) * 0.12, part[column], FIT_MARKER[fit] + '-', ms=5, lw=1, color=FIT_COLOR[fit], label=FIT_LABEL[fit])
        if column == 'best-fit shift':
            ax.axhspan(-SHIFT_LIMIT, SHIFT_LIMIT, color='#e9ecef', zorder=0); ax.axhline(0, color='grey', lw=0.8)
        else:
            ax.axhline(1, color='grey', lw=0.8)
        ax.grid(axis='y', alpha=0.3)
        if col == 0:
            ax.set_ylabel(ylabel)
    axes[0][col].set_title(FAMILY_LABEL[family], fontsize=10)
    axes[-1][col].set_xticks(x, [f'{N_FLOATED[v]}\n{STEP_LABEL[v]}' for v in VARIANTS], fontsize=7, rotation=35, ha='right')
    axes[-1][col].set_xlabel('floated modeling knobs (cumulative)')
low, high = axes[0][0].get_ylim()
axes[0][0].set_ylim(min(low, -1.5 * SHIFT_LIMIT), max(high, 1.5 * SHIFT_LIMIT))
axes[0][0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / 'headline_shift_sigma.pdf'); plt.show()

The posterior $F_A(Q^2)$ of every variant divided by the `n5` median: median (line) and 68 %
band of `n5` (grey) and of the last finished step (colored), to check that nothing happens away
from $Q^2_{\rm ref}$.

In [ ]:
fig, axes = plt.subplots(len(FITS), len(families), figsize=(4.3 * len(families), 2.9 * len(FITS)), sharex=True, sharey=True, squeeze=False)
for col, family in enumerate(families):
    for row, fit in enumerate(FITS):
        ax = axes[row][col]
        ref = BANDS.get((family, REF, fit))
        if ref is None:
            ax.text(0.5, 0.5, f'{REF} not finished', transform=ax.transAxes, ha='center', color='grey'); continue
        done = [v for v in VARIANTS if (family, v, fit) in BANDS]
        with np.errstate(divide='ignore', invalid='ignore'):
            ax.fill_between(Q2, ref['q84'] / ref['median'], ref['q16'] / ref['median'], color='0.85', lw=0, label=f'{REF} 68 %')
            last = BANDS[family, done[-1], fit]
            if done[-1] != REF:
                ax.fill_between(Q2, last['q84'] / ref['median'], last['q16'] / ref['median'], facecolor='none',
                                edgecolor=VARIANT_COLOR[done[-1]], hatch='///', lw=0.6, label=f'{done[-1]} 68 %')
            for v in done:
                ax.plot(Q2, BANDS[family, v, fit]['median'] / ref['median'], color=VARIANT_COLOR[v], lw=1.6 if v == REF else 1.1,
                        label=f'{v}: {STEP_LABEL[v]}')
        ax.axvline(Q2_REFERENCE, color='grey', lw=0.8, ls=':')
        ax.text(0.03, 0.05, FIT_LABEL[fit], transform=ax.transAxes, fontsize=8)
        if col == 0:
            ax.set_ylabel(r'$F_A$ / median of n5')
        if row == len(FITS) - 1:
            ax.set_xlabel(r'$Q^2$ [GeV$^2$]')
    axes[0][col].set_title(FAMILY_LABEL[family], fontsize=10)
handles, labels = max((ax.get_legend_handles_labels() for ax in axes.ravel()), key=lambda hl: len(hl[0]))
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=7, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig(FIG_DIR / 'fa_ratio_to_n5.pdf'); plt.show()

## 4. Fitted parameters and knob pulls

Top table: the independent axial parameters ($M_A$, or $a_1, a_2$) per variant, with the shift of
the profile best fit in units of the `n5` 68 % half-width.

Bottom table and figure: every floated knob per variant, as the profile best fit, the posterior
median and 68 % interval, and the 68 % width over the width of its prior (drawn with the same
truncation as the XML `restrict`). On Asimov data a width ratio near 1 means the data do not
constrain the knob, so floating it can only reproduce the covariance treatment. The two-point
switches live on $[0,1]$ and cannot exceed `PULL_LIMIT` by construction; for them read the median
against the prior median (about 0.4) instead.

In [ ]:
def knob_arrays(result):
    # (labels, posterior samples, prior samples, profile best fit) of the floated knobs
    spec = result['spec']
    if 'ma_joint_samples' in result:
        return (list(spec.profile_labels[1:]), result['ma_joint_samples'][:, 1:],
                result['ma_joint_prior_samples'][:, 1:], result['ma_joint_profile'][1:])
    n_axial = len(result['joint_indices'])
    return (list(spec.nuisance_labels), result['zexp_nuisance_samples'][:, n_axial:],
            result['zexp_nuisance_prior_samples'][:, n_axial:], result['zexp_nuisance_profile'][n_axial:])


param_rows, pull_rows = [], []
for (family, v, fit), result in RESULTS.items():
    summary = result['summary'].iloc[result['joint_indices']]
    ref_result = RESULTS.get((family, REF, fit))
    ref = ref_result['summary'].iloc[ref_result['joint_indices']] if ref_result is not None else None
    for name, row in summary.iterrows():
        half = 0.5 * (row['q84'] - row['q16'])
        ref_half = 0.5 * (ref.loc[name, 'q84'] - ref.loc[name, 'q16']) if ref is not None else np.nan
        param_rows.append(dict(family=family, fit=fit, parameter=name, variant=v, best_fit=row['profile_best_fit'],
                               median=row['posterior_median'], half_width=half, width_ratio=half / ref_half,
                               best_fit_shift=(row['profile_best_fit'] - ref.loc[name, 'profile_best_fit']) / ref_half if ref is not None else np.nan))
    labels, post, prior, best = knob_arrays(result)
    q16, q50, q84 = np.percentile(post, [16, 50, 84], axis=0)
    p16, p50, p84 = np.percentile(prior, [16, 50, 84], axis=0)
    for i, label in enumerate(labels):
        pull_rows.append(dict(family=family, fit=fit, knob=label, variant=v, added=label in LADDER_KNOBS, best_fit=best[i],
                              median=q50[i], q16=q16[i], q84=q84[i], prior_median=p50[i],
                              width_over_prior=(q84[i] - q16[i]) / (p84[i] - p16[i])))
PARAMS, PULLS = pd.DataFrame(param_rows), pd.DataFrame(pull_rows)

values = ['best_fit', 'median', 'half_width', 'best_fit_shift']
param_table = PARAMS.pivot(index=['family', 'fit', 'parameter'], columns='variant', values=values)
param_table = param_table.reindex(columns=pd.MultiIndex.from_product([values, VARIANTS])).dropna(axis=1, how='all')
display(param_table.style.format('{:.3f}', na_rep=''))

values = ['best_fit', 'median', 'width_over_prior']
pull_table = PULLS.pivot(index=['family', 'fit', 'knob'], columns='variant', values=values)
order = pd.MultiIndex.from_product([FAMILIES, FITS, BASELINE + LADDER_KNOBS], names=pull_table.index.names)
pull_table = pull_table.reindex(index=order, columns=pd.MultiIndex.from_product([values, VARIANTS])).dropna(how='all').dropna(axis=1, how='all')
print('knob pulls: profile best fit, posterior median, 68% width over the prior width (1 = unconstrained by the data)')
display(pull_table.style.format('{:.2f}', na_rep=''))

knobs = BASELINE + LADDER_KNOBS
y = np.arange(len(knobs))[::-1]
fig, axes = plt.subplots(len(FITS), len(families), figsize=(4.3 * len(families), 4.3 * len(FITS)), sharex=True, sharey=True, squeeze=False)
for col, family in enumerate(families):
    for row, fit in enumerate(FITS):
        ax = axes[row][col]
        for k, v in enumerate(VARIANTS):
            part = PULLS[(PULLS.family == family) & (PULLS.fit == fit) & (PULLS.variant == v)].set_index('knob').reindex(knobs)
            if part['median'].isna().all():
                continue
            offset = (k - (len(VARIANTS) - 1) / 2) * 0.13
            ax.errorbar(part['median'], y - offset, xerr=[part['median'] - part['q16'], part['q84'] - part['median']],
                        fmt='o', ms=3, color=VARIANT_COLOR[v], capsize=1.5, lw=0.9, label=f'{v}: {STEP_LABEL[v]}')
            ax.plot(part['best_fit'], y - offset, 'x', ms=4, color=VARIANT_COLOR[v])
        ax.axvline(0, color='grey', lw=0.8)
        for limit in (-PULL_LIMIT, PULL_LIMIT):
            ax.axvline(limit, color='grey', lw=0.8, ls=':')
        ax.axhline(y[len(BASELINE) - 1] - 0.5, color='grey', lw=0.8, ls='--')      # production knobs above, added below
        ax.text(0.03, 0.03, FIT_LABEL[fit], transform=ax.transAxes, fontsize=8)
        ax.grid(axis='x', alpha=0.3)
    axes[0][col].set_title(FAMILY_LABEL[family], fontsize=10)
axes[0][0].set_yticks(y, [short_of(knob) for knob in knobs])
for ax in axes[:, 0]:
    ax.tick_params(axis='y', labelsize=8)
fig.supxlabel(r'pull [prior $\sigma$]  (point: posterior median and 68 %, cross: profile best fit)', y=0.045, fontsize=9)
handles, labels = max((ax.get_legend_handles_labels() for ax in axes.ravel()), key=lambda hl: len(hl[0]))
fig.legend(handles, labels, loc='lower center', ncol=len(VARIANTS), fontsize=7, frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 1)); fig.savefig(FIG_DIR / 'knob_pulls.pdf'); plt.show()

## 5. $\chi^2$ bookkeeping

Same split as notebook 3: $\chi^2_{\rm tot}=\chi^2_{\rm stat}+\chi^2_{\rm syst}+\chi^2_{\rm pull}$, with
$\chi^2_{\rm stat}$ the statistical residual of the constrained best fit, $\chi^2_{\rm syst}$ the squared
pulls of the covariance modes (`covmode_pulls`) and $\chi^2_{\rm pull}$ the squared best-fit values
of the Gaussian-constrained parameters from `profile.log`. All added knobs have Gaussian priors,
so $\nu=N_{\rm bins}$ does not change along the ladder. In the linear limit $\chi^2_{\rm tot}$ is
constant and $\chi^2$ only moves from the syst column to the pull column; $\Delta\chi^2_{\rm tot}$
relative to `n5` is therefore a direct measure of how non-linear the added knobs are.

For a CNP fit (output tree with `cnp` in its name, or `AXIAL_MASS_METRIC=CNP`) the statistical variance is $3/(1/d+2/\mu)$ with $\mu$ the best fit *before* the covariance-mode shift, which `PROfile.root` does not store: $\chi^2_{\rm stat}$ is then defined as $\chi^2_{\rm data}-\chi^2_{\rm syst}$ and the closure check is a coarse guard (same sum with $\mu$ reconstructed from `covmode_bin_pulls` and the pre-fit band, tolerance 1.5 instead of 0.5; `python/scripts/chi2_split.py`).


In [ ]:
CHI2_PATTERN = re.compile(r'Global Best Fit chi(?:\^?2|²):\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)')
PARAM_PATTERN = re.compile(r'\|\|\s+(\w+)\s+:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)')
CHANNEL = 'nu_uBooNE_numuCC1p'
from chi2_split import TOLERANCE, fit_metric, stat_syst_split  # stat/syst split, Neyman or CNP
METRIC = fit_metric(OUTPUT_ROOT)


def read_fit_stats(log_file):
    if not log_file.is_file():
        return np.nan, {}
    text = log_file.read_text(errors='replace')
    chi2 = CHI2_PATTERN.findall(text)
    final_block = text.rsplit('Global Best Fit chi^2:', 1)[-1]
    return (float(chi2[-1]) if chi2 else np.nan), {name: float(value) for name, value in PARAM_PATTERN.findall(final_block)}


rows = []
for (family, v, fit) in RESULTS:
    chi2_tot, parameters = read_fit_stats(fit_dir(family, v, fit) / 'profile.log')
    pull = sum(value ** 2 for value in parameters.values()) if parameters else np.nan
    split = stat_syst_split(profile_root(family, v, fit), chi2_tot - pull, METRIC)
    stat, syst = split['stat'], split['syst']
    rows.append(dict(family=family, fit=fit, variant=v, n_parameters=len(parameters), chi2_stat=stat, chi2_syst=syst,
                     chi2_pull=pull, chi2_tot=chi2_tot, split_mismatch=split['mismatch']))
CHI2 = pd.DataFrame(rows).set_index(['family', 'fit', 'variant']).sort_index()
reference = CHI2['chi2_tot'].xs(REF, level='variant') if REF in CHI2.index.get_level_values('variant') else pd.Series(dtype=float)
CHI2['delta_chi2_tot'] = CHI2['chi2_tot'] - reference.reindex(CHI2.index.droplevel('variant')).to_numpy()
expected = {v: N_FLOATED[v] for v in VARIANTS}
wrong = [(key, n) for key, n in CHI2['n_parameters'].items() if n and n not in (expected[key[2]] + 1, expected[key[2]] + 2)]
if wrong:
    print('warning: unexpected number of fitted parameters in profile.log (axial parameters + floated knobs):', wrong)
if (CHI2['split_mismatch'].abs() > TOLERANCE[METRIC]).any():
    print(f'warning: chi2 split closure ({METRIC}) off by more than {TOLERANCE[METRIC]} in the rows with a large split_mismatch')
display(CHI2.reindex(ORDER).dropna(how='all').style.format({c: '{:.3f}' for c in CHI2.columns if c != 'n_parameters'}, na_rep=''))

## 6. Verdict

The decision rule applied per family and fit, over the finished steps: the largest best-fit shift
of $F_A(Q^2_{\rm ref})$ along the ladder and the largest pull (posterior median or best fit) of an
added knob, two-point switches excluded. `complete` says whether every step has finished; a
verdict on an incomplete ladder is provisional. The table is written to
`tables/nuisance_ladder_study/summary.csv`, the per-step numbers to `steps.csv`.

In [ ]:
rows = []
for family in FAMILIES:
    for fit in FITS:
        done = [v for v in VARIANTS if (family, v, fit) in RESULTS]
        if REF not in done or len(done) < 2:
            continue
        steps = FA.loc[(family, fit)].reindex([v for v in done if v != REF])
        pulls = PULLS[(PULLS.family == family) & (PULLS.fit == fit) & PULLS.added & ~PULLS.knob.isin(TWO_POINT)]
        largest = pulls.assign(size=pulls[['median', 'best_fit']].abs().max(axis=1)).sort_values('size').tail(1)
        row = dict(family=family, fit=fit, complete=len(done) == len(VARIANTS), last_step=done[-1],
                   max_shift=steps['best-fit shift'].abs().max(), at_step=steps['best-fit shift'].abs().idxmax(),
                   max_shift_any_Q2=steps['max |shift|'].max(), max_sigma_ratio=steps['sigma ratio'].max(),
                   max_pull=largest['size'].iloc[0] if len(largest) else np.nan,
                   max_pull_knob=f"{largest['knob'].iloc[0]} ({largest['variant'].iloc[0]})" if len(largest) else '',
                   max_delta_chi2=CHI2.loc[(family, fit)]['delta_chi2_tot'].abs().max())
        row['shift_ok'] = row['max_shift'] < SHIFT_LIMIT
        row['pull_ok'] = not (row['max_pull'] >= PULL_LIMIT)
        row['keep_n5'] = row['shift_ok'] and row['pull_ok']
        rows.append(row)
SUMMARY = pd.DataFrame(rows)
if SUMMARY.empty:
    print(f'no family has a finished {REF} and at least one further step yet')
else:
    SUMMARY = SUMMARY.set_index(['family', 'fit'])
    display(SUMMARY.style.format({c: '{:.3f}' for c in SUMMARY.columns if SUMMARY[c].dtype.kind == 'f'}, na_rep=''))
    SUMMARY.to_csv(TABLE_DIR / 'summary.csv', float_format='%.5g')
    FA.reindex(ORDER).dropna(how='all').join(CHI2[['chi2_stat', 'chi2_syst', 'chi2_pull', 'chi2_tot', 'delta_chi2_tot']]).to_csv(TABLE_DIR / 'steps.csv', float_format='%.5g')
    PULLS.to_csv(TABLE_DIR / 'knob_pulls.csv', index=False, float_format='%.5g')
    verdict = 'keep the production five' if SUMMARY['keep_n5'].all() else 'the rule is violated in the rows with keep_n5 = False'
    status = 'all steps finished' if SUMMARY['complete'].all() and len(SUMMARY) == len(FAMILIES) * len(FITS) else 'PROVISIONAL, fits still missing'
    print(f'verdict ({status}): {verdict}')